<a href="https://colab.research.google.com/github/zorGizem/Erken-Evre-Alzhemir-Tespiti/blob/main/notebooks/08_slice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Kritik Kesit Çıkarma ve Bounding Box Uygulaması
Bu aşamada, 3D MRI hacimlerinden Alzheimer teşhisi için en yüksek bilgi yoğunluğuna sahip olan bölgeler seçilerek 2D kesitler (slices) elde edilmiştir.

**İşlemin Teknik Detayları**

**Kritik Bölge Seçimi:** MNI koordinat sistemi kullanılarak hipokampüs, entorhinal korteks ve ventriküller gibi atrofinin en belirgin izlendiği koronal, aksiyal ve sagital düzlemlerden toplam 15 stratejik kesit seçilmiştir.

**Bounding Box (Sınır Kutusu):** Her hasta için beyin dokusunu çevreleyen en küçük dikdörtgen alan (Bounding Box) hesaplanarak görüntüler kırpılmıştır. Bu sayede görüntüdeki gereksiz siyah boşluklar atılmış ve modelin sadece anlamlı beyin dokusuna odaklanması sağlanmıştır.

**Boyut Tutarlılığı:** Tüm kesitler, veri setindeki boyut farklılıkları elenerek derin öğrenme modelinin girdi katmanıyla tam uyumlu ve homojen bir yapıya getirilmiştir.

In [ ]:
import numpy as np
import nibabel as nib
print("Kütüphaneler hazır")

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU Aktif: {torch.cuda.get_device_name(0)}")
else:
    print(" GPU bulunamadı!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

CONFIG_CN = {
    "kategori"    : "CN",
    "kaynak": f'/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/Normalizasyonn/Normalizasyon_CN',
    "hedef" : f'/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/Slicee/Slice_CN',
}

CONFIG_EMCI = {
    "kategori"    : "EMCI",
    "kaynak": f'/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/Normalizasyonn/Normalizasyon_EMCI',
    "hedef" : f'/content/drive/MyDrive/GizemZor/YAZILIM_TASARIMI/Bitirme/Dataset/Slicee/Slice_EMCI',
}

# MNI uzayında kritik bölge koordinatları (mm cinsinden)
MNI_KOORDINATLAR = {
    "koronal": [-20, -24, -28, -32, -36],  # Hipokampüs + Entorhinal korteks
    "aksiyal": [10, 15, 20, 25, 30],       # Ventriküller + Genel atrofi
    "sagital": [-6, -3, 0, 3, 6],          # Korpus kallozum + Medial yapılar
}

In [ ]:
import os
import shutil
import numpy as np
import nibabel as nib

def mni_mm_to_voxel(affine, mni_mm):
    """MNI mm koordinatını voksel indeksine çevirir."""
    inv_affine = np.linalg.inv(affine)
    voxel = np.dot(inv_affine, np.array([*mni_mm, 1]))
    return int(round(voxel[0])), int(round(voxel[1])), int(round(voxel[2]))

def get_bounding_box_limits(data):
    """0 olmayan (beyin) piksellerinin sınırlarını (Bounding Box) bulur."""
    coords = np.argwhere(data != 0)
    if coords.size == 0:
        return None
    mn = coords.min(axis=0)
    mx = coords.max(axis=0)
    return mn, mx

def batch_nifti_slice_extraction(config, koordinatlar, batch_size=20):
    source_base = config["kaynak"]
    output_base = config["hedef"]
    etiket = config["kategori"]

    local_in = "/content/batch_in_slice"
    local_out = "/content/batch_out_slice"

    print(f"\n {etiket} NIfTI KESİT ÇIKARMA Başlıyor (Grup: {batch_size} | Bounding Box )")

    # 1. Analiz Aşaması
    all_pending = []
    subjects = sorted([s for s in os.listdir(source_base) if os.path.isdir(os.path.join(source_base, s))])

    toplam_dosya = 0
    atlanan_dosya = 0
    basarili_dosya = 0

    print(" Klasörler taranıyor, lütfen bekleyin...", end="\r")

    for subject_id in subjects:
        subject_path = os.path.join(source_base, subject_id)
        seanslar = [d for d in os.listdir(subject_path) if os.path.isdir(os.path.join(subject_path, d))]

        for seans in seanslar:
            seans_yolu = os.path.join(subject_path, seans)
            nifti_files = [f for f in os.listdir(seans_yolu) if f.endswith('.nii.gz')]

            if nifti_files:
                toplam_dosya += 1
                f = nifti_files[0]
                rel_path = os.path.join(subject_id, seans)
                cikti_klasoru = os.path.join(output_base, rel_path)

                # Klasör var mı ve içinde 15 kesit var mı?
                if not os.path.exists(cikti_klasoru) or len(os.listdir(cikti_klasoru)) < 15:
                    all_pending.append((os.path.join(seans_yolu, f), rel_path, f, subject_id))
                else:
                    atlanan_dosya += 1

    kalan_dosya = len(all_pending)
    print(" "*50, end="\r")
    print(f" KLASÖR ANALİZİ ({etiket}):")
    print(f"    Toplam Bulunan Dosya : {toplam_dosya}")
    print(f"    Zaten İşlenmiş Olan  : {atlanan_dosya}")
    print(f"    Şimdi İşlenecek Olan : {kalan_dosya}")
    print("-" * 40)

    if not all_pending:
        print(f" İşlenecek yeni dosya yok.")
    else:
        # 2. Döngü
        for i in range(0, kalan_dosya, batch_size):
            batch = all_pending[i:i+batch_size]
            shutil.rmtree(local_in, ignore_errors=True)
            shutil.rmtree(local_out, ignore_errors=True)
            os.makedirs(local_in, exist_ok=True)
            os.makedirs(local_out, exist_ok=True)

            batch_subjects = sorted(list(set([item[3] for item in batch])))
            print(f"\n Grup {i//batch_size + 1} Başlıyor...")
            print(f" İşlenen Hastalar: {', '.join(batch_subjects)}")

            mapping = {}
            for full_src, rel_p, fname, sid in batch:
                unique_name = rel_p.replace("/", "_") + "___" + fname
                local_girdi = os.path.join(local_in, unique_name)
                shutil.copy2(full_src, local_girdi)
                mapping[unique_name] = (rel_p, fname, local_girdi)

            print(f" Kesitler alınıyor ve Bounding Box uygulanıyor... ", end="", flush=True)

            try:
                for u_name, (rel_p, fname, l_girdi) in mapping.items():
                    img = nib.load(l_girdi)
                    veri = img.get_fdata(dtype=np.float32)
                    affine = img.affine
                    header = img.header

                    # Bounding Box sınırlarını bul (0 olmayan alan)
                    limits = get_bounding_box_limits(veri)
                    if limits is None: continue
                    mn, mx = limits

                    # Her hasta için yerelde geçici klasör
                    temp_patient_dir = os.path.join(local_out, rel_p.replace("/", "_"))
                    os.makedirs(temp_patient_dir, exist_ok=True)

                    for duzlem, mni_listesi in koordinatlar.items():
                        for j, mni_mm in enumerate(mni_listesi):

                            if duzlem == "koronal":
                                _, vy, _ = mni_mm_to_voxel(affine, [0, mni_mm, 0])
                                vy = np.clip(vy, 0, veri.shape[1] - 1)
                                # Kırpılmış kesit (x ve z eksenlerinde)
                                kesit_veri = veri[mn[0]:mx[0]+1, vy, mn[2]:mx[2]+1]
                                kesit_veri = kesit_veri[:, np.newaxis, :] # (x, 1, z)

                            elif duzlem == "aksiyal":
                                _, _, vz = mni_mm_to_voxel(affine, [0, 0, mni_mm])
                                vz = np.clip(vz, 0, veri.shape[2] - 1)
                                # Kırpılmış kesit (x ve y eksenlerinde)
                                kesit_veri = veri[mn[0]:mx[0]+1, mn[1]:mx[1]+1, vz]
                                kesit_veri = kesit_veri[:, :, np.newaxis] # (x, y, 1)

                            elif duzlem == "sagital":
                                vx, _, _ = mni_mm_to_voxel(affine, [mni_mm, 0, 0])
                                vx = np.clip(vx, 0, veri.shape[0] - 1)
                                # Kırpılmış kesit (y ve z eksenlerinde)
                                kesit_veri = veri[vx, mn[1]:mx[1]+1, mn[2]:mx[2]+1]
                                kesit_veri = kesit_veri[np.newaxis, :, :] # (1, y, z)

                            # NIfTI Kaydı ve İsimlendirme
                            slice_name = f"{duzlem}_{j+1:02d}.nii.gz"
                            slice_img = nib.Nifti1Image(kesit_veri, affine, header)
                            nib.save(slice_img, os.path.join(temp_patient_dir, slice_name))

                print(" Bitti")

                print(f" Drive'a aktarılıyor... ", end="", flush=True)
                for u_name, (rel_p, fname, _) in mapping.items():
                    l_patient_res = os.path.join(local_out, rel_p.replace("/", "_"))
                    d_target_dir = os.path.join(output_base, rel_p)

                    if os.path.exists(l_patient_res):
                        shutil.copytree(l_patient_res, d_target_dir, dirs_exist_ok=True)
                        basarili_dosya += 1

                su_an_biten = min(i+batch_size, kalan_dosya)
                print(f" {su_an_biten}/{kalan_dosya} yeni dosya kaydedildi.")

            except Exception as e:
                print(f" Grup Hatası: {str(e)}")

    # 3. Özet
    print(f"\n{etiket} Kesit Çıkarma Özeti:")
    print(f"Toplam   : {toplam_dosya}")
    print(f"Başarılı : {basarili_dosya}")
    print(f"Atlanan  : {atlanan_dosya}")
    print(f"{etiket} grubu tamamlandı.")

In [ ]:
batch_nifti_slice_extraction(CONFIG_CN, MNI_KOORDINATLAR)

In [ ]:
batch_nifti_slice_extraction(CONFIG_EMCI, MNI_KOORDINATLAR)